In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/transactions_rules.csv')

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,...,balance_diff,is_high_risk_type,rule_high_value,rule_high_risk_type,rule_account_emptied,rule_amount_spike,rule_velocity,rule_score,rule_based_alert,rule_alert_strict
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,...,9839.64,0,0,0,0,0,0,0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,...,1864.28,0,0,0,0,0,0,0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,...,181.00,1,0,1,1,0,0,2,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,...,181.00,1,0,1,1,0,0,2,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,...,11668.14,0,0,0,0,0,0,0,0,0


In [2]:
features = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'oldbalanceDest',
    'newbalanceDest',
    'txn_count_sender',
    'total_sent',
    'avg_sent',
    'txn_per_step',
    'amount_deviation',
    'balance_diff',
    'is_high_risk_type',
    'rule_score',
    'rule_based_alert'
]

target = 'isFraud'

X = df[features]
y = df[target]

In [3]:
fraud_df = df[df['isFraud'] == 1]
non_fraud_df = df[df['isFraud'] == 0].sample(n=100000, random_state=42)

model_df = pd.concat([fraud_df, non_fraud_df], axis=0).sample(frac=1, random_state=42)

model_df['isFraud'].value_counts()

isFraud
0    100000
1      8213
Name: count, dtype: int64

In [4]:
X = model_df[features]
y = model_df[target]

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((86570, 14), (21643, 14))

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

In [7]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20000
           1       0.98      0.99      0.98      1643

    accuracy                           1.00     21643
   macro avg       0.99      0.99      0.99     21643
weighted avg       1.00      1.00      1.00     21643

[[19960    40]
 [   16  1627]]
ROC AUC: 0.9995508064516127


In [8]:
importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values(by='importance', ascending=False)

importance

,feature,importance
10,balance_diff,2.288494e-01
13,rule_based_alert,1.620678e-01
1,oldbalanceOrg,1.498846e-01
12,rule_score,1.475053e-01
7,avg_sent,6.423380e-02
0,amount,5.534626e-02
11,is_high_risk_type,5.355174e-02
2,newbalanceOrig,5.129851e-02
6,total_sent,4.340606e-02
4,newbalanceDest,2.630363e-02


In [9]:
importance.to_csv('../reports/model_feature_importance.csv', index=False)

In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

metrics = pd.DataFrame({
    'model': ['Random Forest'],
    'accuracy': [accuracy_score(y_test, y_pred)],
    'precision': [precision_score(y_test, y_pred)],
    'recall': [recall_score(y_test, y_pred)],
    'f1_score': [f1_score(y_test, y_pred)],
    'roc_auc': [roc_auc_score(y_test, y_proba)]
})

metrics

,model,accuracy,precision,recall,f1_score,roc_auc
0,Random Forest,0.997413,0.976005,0.990262,0.983082,0.999551


In [11]:
metrics.to_csv('../reports/model_metrics.csv', index=False)

In [12]:
import joblib

joblib.dump(rf_model, '../models/random_forest_aml_model.pkl')


['../models/random_forest_aml_model.pkl']

## Model Training Summary

A Random Forest classification model was trained to support AML alert prioritisation.

Because fraud cases are highly imbalanced in the full dataset, the training set used all fraud cases and a sampled set of non-fraud transactions. The model uses transaction behaviour, account activity, rule-based signals, and balance movement features.

The purpose of this model is not only to classify fraud, but to support risk scoring and prioritisation of suspicious alerts for investigation.